Pipeline Parameters

In [0]:
dbutils.widgets.text("catalog_name", "workspace")
dbutils.widgets.text("schema_name", "bronze")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook")
dbutils.widgets.text("table_name", "all")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name  = dbutils.widgets.get("schema_name")
base_path    = dbutils.widgets.get("base_path")
table_name   = dbutils.widgets.get("table_name")

print("=" * 50)
print("PIPELINE PARAMETERS")
print("=" * 50)
print(f"Catalog   : {catalog_name}")
print(f"Schema    : {schema_name}")
print(f"Base Path : {base_path}")
print(f"Table     : {table_name}")
print("=" * 50)

Read Active tables from metadata

In [0]:
active_tables = spark.sql(f"""
    SELECT table_name, file_name
    FROM {catalog_name}.{schema_name}.pipeline_control
    WHERE active_flag = 'Y'
    ORDER BY table_name
""")

print("=" * 50)
print("ACTIVE TABLES TO PROCESS")
print("=" * 50)
active_tables.show()
print(f"Total: {active_tables.count()} tables")

Extract from surce and write to raw

In [0]:
from datetime import datetime

exec_time = datetime.now()
year  = exec_time.strftime("%Y")
month = exec_time.strftime("%m")
day   = exec_time.strftime("%d")

success_tables = []
failed_tables  = []

print("=" * 50)
print("STARTING RAW ZONE LOAD")
print("=" * 50)

for row in active_tables.collect():
    tbl = row["table_name"]   # PascalCase e.g. Album, InvoiceLine
    
    try:
        print(f"\n📥 Processing: {tbl}")
        
        # Step 1 — Re-read from source (temp views don't survive across Job tasks)
        df = spark.read.table(f"chinook_connection_catalog.chinook.{tbl}")
        source_count = df.count()
        print(f"   Source rows : {source_count}")
        
        # Step 2 — Build dynamic file path with date partitioning
        file_path = f"{base_path}/{tbl}/{year}/{month}/{day}/{tbl}.parquet"
        print(f"   Writing to  : {file_path}")
        
        # Step 3 — Write Parquet to Raw Volume (never overwrite existing runs)
        df.write.mode("overwrite").parquet(file_path)
        
        # Step 4 — Verify row count matches
        target_count = spark.read.parquet(file_path).count()
        status = "SUCCESS" if source_count == target_count else "FAILED"
        print(f"   Target rows : {target_count}")
        print(f"   Status      : {status}")
        
        # Step 5 — Log to execution_log (child metadata table)
        log_df = spark.createDataFrame([{
            "table_name"       : tbl,
            "execution_time"   : exec_time,
            "status"           : status,
            "source_row_count" : source_count,
            "target_row_count" : target_count,
            "file_location"    : file_path,
            "created_date"     : exec_time.date()
        }])
        
        log_df.write.format("delta").mode("append") \
              .saveAsTable(f"{catalog_name}.{schema_name}.execution_log")
        
        success_tables.append(tbl)
        print(f"   ✅ Logged to execution_log")
        
    except Exception as e:
        failed_tables.append(tbl)
        print(f"   ❌ FAILED | {str(e)}")
        
        # Log the failure too
        try:
            fail_df = spark.createDataFrame([{
                "table_name"       : tbl,
                "execution_time"   : exec_time,
                "status"           : "FAILED",
                "source_row_count" : 0,
                "target_row_count" : 0,
                "file_location"    : "",
                "created_date"     : exec_time.date()
            }])
            fail_df.write.format("delta").mode("append") \
                   .saveAsTable(f"{catalog_name}.{schema_name}.execution_log")
        except:
            print(f"   ⚠️ Could not log failure for {tbl}")

Summary

In [0]:
print("=" * 50)
print("RAW ZONE LOAD SUMMARY")
print("=" * 50)
print(f"✅ Successful : {len(success_tables)}")
print(f"❌ Failed     : {len(failed_tables)}")
print("\nSuccessful tables:")
for t in success_tables:
    print(f"  - {t}")
if failed_tables:
    print("\nFailed tables:")
    for t in failed_tables:
        print(f"  - {t}")
print("=" * 50)

Validate: check execution_log was written

In [0]:
print("Validating execution_log entries...")
spark.sql(f"""
    SELECT table_name, status, source_row_count, target_row_count, 
           file_location, execution_time
    FROM {catalog_name}.{schema_name}.execution_log
    ORDER BY execution_time DESC
""").show(truncate=False)

Validate: spot check one Raw file

In [0]:
# Spot check Album parquet was written correctly
test_path = f"{base_path}/Album/{year}/{month}/{day}/Album.parquet"
test_df = spark.read.parquet(test_path)
print(f"Album parquet row count: {test_df.count()}")
test_df.show(5)